# Compound Validation Tutorial

## Introduction

This notebook demonstrates how to validate compound structures using the `CompoundValidator` class, which implements the Odele and Macchietto structural feasibility rules for group contribution methods.

## What is Compound Validation?

In Computer-Aided Molecular Design (CAMD), we need to ensure that generated molecular structures are chemically feasible. The CompoundValidator checks three key rules:

1. **Standalone Molecule Rule**: Groups with no connections (NOC=0) like H₂O, CO₂ can only exist alone
2. **Octet Rule**: Valency satisfaction - `2*m = Σ((2 - noc_i) * x_i)`
3. **Connectivity Rule**: Sufficient connections - `Σ(x_j where j≠i) ≥ (noc_i - 2) * x_i + 2`

Where:
- `m`: molecular type (1=acyclic, 0=monocyclic, -1=bicyclic)  
- `noc_i`: number of connections for group i
- `x_i`: occurrence count of group i

## References
- [Odele & Macchietto (1993)](https://doi.org/10.1016/0378-3812(93)87127-M) - Structural feasibility rules
- [Constantinou & Gani (1994)](https://aiche.onlinelibrary.wiley.com/doi/abs/10.1002/aic.690401011) - Group contribution method

## Setup and Imports

In [1]:
# Import necessary libraries
import sys
import pandas as pd
import os
from pathlib import Path

# Add src to path for imports - fix the path resolution
current_dir = Path(os.getcwd())
if current_dir.name == 'tutorials':
    # If running from tutorials directory
    src_path = current_dir.parent / "src"
else:
    # If running from project root
    src_path = current_dir / "src"

sys.path.insert(0, str(src_path))
print(f"Added to path: {src_path}")

# Now import the CompoundValidator
try:
    from compound_validator import CompoundValidator, validate_compound, filter_valid_compounds
    print("✓ Successfully imported CompoundValidator")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print(f"Current working directory: {os.getcwd()}")
    print(f"Python path: {sys.path[:3]}...")

# Initialize the validator
try:
    validator = CompoundValidator()
    print("\n=== CompoundValidator Initialized ===")
    print(f"Molecular type: {validator.molecular_type} (1=acyclic, 0=monocyclic, -1=bicyclic)")
    print(f"Data path: {validator.data_path}")
    print(f"NOC data loaded: {len(validator.noc_data)} groups")
except Exception as e:
    print(f"✗ Failed to initialize CompoundValidator: {e}")

Added to path: /Users/arnaou/projects/dtu_chemeng/camd_iss/src
✓ Successfully imported CompoundValidator
Loaded noc data: 119 groups

=== CompoundValidator Initialized ===
Molecular type: 1 (1=acyclic, 0=monocyclic, -1=bicyclic)
Data path: /Users/arnaou/projects/dtu_chemeng/camd_iss/data/json
NOC data loaded: 119 groups


## Example 1: Basic Compound Validation

Let's start with some simple examples to understand how compound validation works.

In [8]:
# Define some test compounds
test_compounds = {
    'Ethanol': {'CH3': 1, 'CH2': 1, 'OH': 1},      # Valid acyclic molecule
    'Acetone': {'CH3': 1, 'CH3CO': 1},             # Valid acyclic molecule  
    'Water': {'H2O': 1},                           # Valid standalone molecule
    'Invalid_1': {'CH3': 10},                      # Too many CH3 groups
    'Invalid_2': {'OH': 5},                        # Too many OH groups
}

print("=== Basic Compound Validation ===")
print()

for name, compound in test_compounds.items():
    print(f"{name}: {compound}")
    
    # Validate the compound
    is_valid = validator.validate_compound(compound, verbose=True)
    
    print(f"Result: {'✓ VALID' if is_valid else '✗ INVALID'}")
    print("-" * 60)
    print()

=== Basic Compound Validation ===

Ethanol: {'CH3': 1, 'CH2': 1, 'OH': 1}
Octet rule satisfied: valency sum = 2
Connectivity rule satisfied for all groups
Compound {'CH3': 1, 'CH2': 1, 'OH': 1} is structurally valid
Result: ✓ VALID
------------------------------------------------------------

Acetone: {'CH3': 1, 'CH3CO': 1}
Octet rule satisfied: valency sum = 2
Connectivity rule satisfied for all groups
Compound {'CH3': 1, 'CH3CO': 1} is structurally valid
Result: ✓ VALID
------------------------------------------------------------

Water: {'H2O': 1}
Compound {'H2O': 1} is a valid standalone molecule
Result: ✓ VALID
------------------------------------------------------------

Invalid_1: {'CH3': 10}
Octet rule violation: valency sum = 10, expected = 2 (for acyclic molecule)
Result: ✗ INVALID
------------------------------------------------------------

Invalid_2: {'OH': 5}
Octet rule violation: valency sum = 5, expected = 2 (for acyclic molecule)
Result: ✗ INVALID
---------------------

## Example 2: Standalone Molecule Rules

Standalone molecules are groups with NOC=0 (no connections). They have special validation rules:

In [10]:
# Standalone molecule test cases
standalone_tests = {
    'Valid - Single water': {'H2O': 1},                    # ✓ Valid: single standalone molecule
    'Invalid - Multiple water': {'H2O': 2},                # ✗ Invalid: count > 1
    'Invalid - Mixed with others': {'H2O': 1, 'CH3': 1},   # ✗ Invalid: mixed with other groups
    'Invalid - Multiple types': {'H2O': 1, 'CO2': 1},      # ✗ Invalid: multiple standalone types
    'Invalid - Single CO2': {'CO2': 1},                      # ✗ Invalid: single CO2 is not a group according to unifac/CG
}

print("=== Standalone Molecule Validation Rules ===")
print()
print("Rules:")
print("1. Single standalone molecule with count=1 is VALID")
print("2. Multiple copies of standalone molecule is INVALID") 
print("3. Standalone molecule mixed with other groups is INVALID")
print("4. Multiple different standalone molecules is INVALID")
print()

for description, compound in standalone_tests.items():
    print(f"{description}: {compound}")
    is_valid = validator.validate_compound(compound, verbose=True)
    print(f"Result: {'✓ VALID' if is_valid else '✗ INVALID'}")
    print("-" * 70)
    print()

=== Standalone Molecule Validation Rules ===

Rules:
1. Single standalone molecule with count=1 is VALID
2. Multiple copies of standalone molecule is INVALID
3. Standalone molecule mixed with other groups is INVALID
4. Multiple different standalone molecules is INVALID

Valid - Single water: {'H2O': 1}
Compound {'H2O': 1} is a valid standalone molecule
Result: ✓ VALID
----------------------------------------------------------------------

Invalid - Multiple water: {'H2O': 2}
Invalid compound: Standalone molecule 'H2O' has count > 1 (2)
Standalone molecules must have count = 1
Result: ✗ INVALID
----------------------------------------------------------------------

Invalid - Mixed with others: {'H2O': 1, 'CH3': 1}
Invalid compound: Standalone molecule 'H2O' mixed with other groups: ['CH3']
Standalone molecules cannot be part of larger structures
Result: ✗ INVALID
----------------------------------------------------------------------

Invalid - Multiple types: {'H2O': 1, 'CO2': 1}
Invali

## Example 3: Molecular Types and Octet Rule

The validator supports different molecular types that affect the octet rule calculation:

In [11]:
# Test the same compound with different molecular types
test_compound = {'CH3': 1, 'CH2': 1, 'OH': 1}  # Ethanol

print("=== Molecular Type Effects on Validation ===")
print(f"Test compound: {test_compound}")
print()
print("Octet rule: 2*m = Σ((2 - noc_i) * x_i)")
print("Where m = molecular type: 1=acyclic, 0=monocyclic, -1=bicyclic")
print()

molecular_types = {
    1: "Acyclic (linear/branched)",
    0: "Monocyclic (one ring)", 
    -1: "Bicyclic (two rings)"
}

for mol_type, description in molecular_types.items():
    print(f"Testing with molecular type {mol_type} ({description}):")
    
    # Create validator with specific molecular type
    validator_typed = CompoundValidator(molecular_type=mol_type)
    
    # Validate the compound
    is_valid = validator_typed.validate_compound(test_compound, verbose=True)
    
    print(f"Result: {'✓ VALID' if is_valid else '✗ INVALID'}")
    print("-" * 60)
    print()

=== Molecular Type Effects on Validation ===
Test compound: {'CH3': 1, 'CH2': 1, 'OH': 1}

Octet rule: 2*m = Σ((2 - noc_i) * x_i)
Where m = molecular type: 1=acyclic, 0=monocyclic, -1=bicyclic

Testing with molecular type 1 (Acyclic (linear/branched)):
Loaded noc data: 119 groups
Octet rule satisfied: valency sum = 2
Connectivity rule satisfied for all groups
Compound {'CH3': 1, 'CH2': 1, 'OH': 1} is structurally valid
Result: ✓ VALID
------------------------------------------------------------

Testing with molecular type 0 (Monocyclic (one ring)):
Loaded noc data: 119 groups
Octet rule violation: valency sum = 2, expected = 0 (for monocyclic molecule)
Result: ✗ INVALID
------------------------------------------------------------

Testing with molecular type -1 (Bicyclic (two rings)):
Loaded noc data: 119 groups
Octet rule violation: valency sum = 2, expected = -2 (for bicyclic molecule)
Result: ✗ INVALID
------------------------------------------------------------



## Example 4: Batch Validation and Statistics

For CAMD applications, you often need to validate many compounds at once:

In [13]:
# Create a batch of compounds for validation
compound_batch = [
    {'H2O': 1},                                # Valid standalone
    {'CH3': 1, 'CH2': 1, 'OH': 1},             # Valid ethanol
    {'CH3': 2, 'CH3CO': 1},                    # Valid acetone
    {'CH3': 1, 'CH2': 2, 'OH': 1},             # Valid propanol
    {'H2O': 2},                                # Invalid - multiple standalone
    {'CH3': 10},                               # Invalid - octet violation
    {'OH': 5},                                 # Invalid - octet violation
    {'CH3': 1, 'CH2': 3, 'H2O': 1, 'CHO': 1}, # Invalid - mixed with standalone
    {'H2O': 1, 'DMSO': 1},                     # Invalid - multiple standalone types
]

print("=== Batch Validation Example ===")
print(f"Validating {len(compound_batch)} compounds...")
print()

# Filter valid and invalid compounds
valid_compounds, invalid_compounds = validator.filter_valid_compounds(compound_batch, verbose=False)

print(f"Results:")
print(f"  Valid compounds: {len(valid_compounds)}")
print(f"  Invalid compounds: {len(invalid_compounds)}")
print()

# Show valid compounds
print("Valid compounds:")
for i, compound in enumerate(valid_compounds, 1):
    print(f"  {i}. {compound}")

print()

# Show invalid compounds  
print("Invalid compounds:")
for i, compound in enumerate(invalid_compounds, 1):
    print(f"  {i}. {compound}")

print()

# Get detailed statistics
stats = validator.get_validation_stats(compound_batch)
print("=== Detailed Statistics ===")
for key, value in stats.items():
    print(f"  {key}: {value}")
    
# Calculate percentages
if stats['total'] > 0:
    valid_pct = (stats['valid'] / stats['total']) * 100
    print(f"\nValidation success rate: {valid_pct:.1f}%")

=== Batch Validation Example ===
Validating 9 compounds...

Results:
  Valid compounds: 3
  Invalid compounds: 6

Valid compounds:
  1. {'H2O': 1}
  2. {'CH3': 1, 'CH2': 1, 'OH': 1}
  3. {'CH3': 1, 'CH2': 2, 'OH': 1}

Invalid compounds:
  1. {'CH3': 2, 'CH3CO': 1}
  2. {'H2O': 2}
  3. {'CH3': 10}
  4. {'OH': 5}
  5. {'CH3': 1, 'CH2': 3, 'H2O': 1, 'CHO': 1}
  6. {'H2O': 1, 'DMSO': 1}

=== Detailed Statistics ===
  total: 9
  valid: 3
  invalid: 6
  standalone_violations: 3
  octet_violations: 3
  connectivity_violations: 0
  missing_data: 0

Validation success rate: 33.3%


## Example 5: Convenience Functions

For quick validation tasks, you can use the standalone convenience functions:

In [14]:
# Using convenience functions for quick validation
print("=== Convenience Functions Example ===")
print()

# Quick single compound validation
compound = {'CH3': 2, 'CH3CO': 1}  # Acetone
print(f"Quick validation of acetone {compound}:")
is_valid = validate_compound(compound, verbose=True)
print(f"Result: {'✓ VALID' if is_valid else '✗ INVALID'}")
print()

# Quick batch filtering
test_compounds = [
    {'CH3': 1, 'CH2': 1, 'OH': 1},   # Valid ethanol
    {'H2O': 1},                      # Valid water
    {'CH3': 10},                     # Invalid
    {'H2O': 1, 'CH3': 1},            # Invalid mixed
]

print("Quick batch filtering:")
print(f"Input: {len(test_compounds)} compounds")
valid_only = filter_valid_compounds(test_compounds, verbose=False)
print(f"Output: {len(valid_only)} valid compounds")
print("Valid compounds:")
for i, compound in enumerate(valid_only, 1):
    print(f"  {i}. {compound}")

=== Convenience Functions Example ===

Quick validation of acetone {'CH3': 2, 'CH3CO': 1}:
Loaded noc data: 119 groups
Octet rule violation: valency sum = 3, expected = 2 (for acyclic molecule)
Result: ✗ INVALID

Quick batch filtering:
Input: 4 compounds
Loaded noc data: 119 groups
Output: 2 valid compounds
Valid compounds:
  1. {'CH3': 1, 'CH2': 1, 'OH': 1}
  2. {'H2O': 1}


## Example 6: Understanding NOC (Number of Connections)

To better understand validation, let's explore the NOC values for different groups:

In [17]:
# Explore NOC values for common groups
print("=== Understanding NOC (Number of Connections) ===")
print()
print("NOC represents how many connections a group can make to other groups:")
print("- NOC = 0: Standalone molecules (H2O, DOH)")
print("- NOC = 1: Terminal groups (CH3, OH)")  
print("- NOC = 2: Linear groups (CH2, O)")
print("- NOC = 3: Branching groups (CH)")
print("- NOC = 4: Quaternary groups (C)")
print()

# Sample some common groups and their NOC values
common_groups = ['H2O', 'DOH', 'CH3', 'OH', 'CH2', 'CH', 'C', 'CH3CO', 'ACH']

print("Common groups and their NOC values:")
print("-" * 40)
for group in common_groups:
    if group in validator.noc_data:
        noc = validator.noc_data[group]
        print(f"{group:8s}: NOC = {noc}")
    else:
        print(f"{group:8s}: NOC = (not found)")

print()
print("NOC interpretation:")
print("- Groups with NOC = 0 are complete molecules")
print("- Groups with NOC = 1 are terminal (end of chain)")
print("- Groups with NOC = 2 can form linear chains")
print("- Groups with NOC ≥ 3 create branching points")

# Show how NOC affects octet rule calculation
print()
print("=== NOC in Octet Rule Calculation ===")
print("Formula: 2*m = Σ((2 - noc_i) * x_i)")
print()

example_compound = {'CH3': 2, 'CH2': 1, 'OH': 1}  # Propanol
print(f"Example: {example_compound}")

total_valency = 0
for group, count in example_compound.items():
    if group in validator.noc_data:
        noc = validator.noc_data[group]
        contribution = (2 - noc) * count
        total_valency += contribution
        print(f"  {group}: NOC={noc}, count={count}, contribution=(2-{noc})*{count} = {contribution}")

print(f"Total valency sum: {total_valency}")
print(f"Expected for acyclic (m=1): 2*1 = 2")
print(f"Match: {'✓ YES' if total_valency == 2 else '✗ NO'}")

=== Understanding NOC (Number of Connections) ===

NOC represents how many connections a group can make to other groups:
- NOC = 0: Standalone molecules (H2O, DOH)
- NOC = 1: Terminal groups (CH3, OH)
- NOC = 2: Linear groups (CH2, O)
- NOC = 3: Branching groups (CH)
- NOC = 4: Quaternary groups (C)

Common groups and their NOC values:
----------------------------------------
H2O     : NOC = 0
DOH     : NOC = 0
CH3     : NOC = 1
OH      : NOC = 1
CH2     : NOC = 2
CH      : NOC = 3
C       : NOC = 4
CH3CO   : NOC = 1
ACH     : NOC = 2

NOC interpretation:
- Groups with NOC = 0 are complete molecules
- Groups with NOC = 1 are terminal (end of chain)
- Groups with NOC = 2 can form linear chains
- Groups with NOC ≥ 3 create branching points

=== NOC in Octet Rule Calculation ===
Formula: 2*m = Σ((2 - noc_i) * x_i)

Example: {'CH3': 2, 'CH2': 1, 'OH': 1}
  CH3: NOC=1, count=2, contribution=(2-1)*2 = 2
  CH2: NOC=2, count=1, contribution=(2-2)*1 = 0
  OH: NOC=1, count=1, contribution=(2-1)*

## Conclusion

The `CompoundValidator` is an essential tool for Computer-Aided Molecular Design that ensures generated molecular structures are chemically feasible. 

### Key Takeaways:

1. **Three Validation Rules**:
   - **Standalone molecules**: NOC=0 groups can only exist alone with count=1
   - **Octet rule**: Ensures proper valency satisfaction 
   - **Connectivity rule**: Ensures sufficient connections between groups

2. **Molecular Types Matter**: Different molecular types (acyclic, monocyclic, bicyclic) have different validation requirements

3. **Batch Processing**: Efficient validation of multiple compounds with detailed statistics

4. **Integration Ready**: Easy to integrate into CAMD workflows for filtering invalid structures

### Best Practices:

- Always validate compounds before property calculations
- Use appropriate molecular type for your application
- Monitor validation statistics to understand failure modes
- Use verbose mode during development for debugging